# ResNet18 오염도 분류 — corrected bbox padding 비교

corrected GT bbox로 생성한 세 crop 데이터셋을 같은 조건으로 순차 학습합니다.

- 입력: `/content/drive/MyDrive/TeamProject/test_dataset/crops_bboxfixed/pad000|pad005|pad010`
- 클래스 인덱스 고정: `0=clean`, `1=outer`, `2=inner`
- 백본: ImageNet 사전학습 ResNet18
- 모델 선택 기준: validation macro F1
- 클래스 불균형 대응: weighted cross entropy
- 출력: `/content/drive/MyDrive/TeamProject/test_dataset/runs/resnet18_dirty3_bboxfixed_padXXX`

세 실험에서 padding 이외의 조건은 동일하게 유지합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. 환경과 입력 데이터 확인

GPU, PyTorch, 데이터 폴더, 클래스별 파일 수를 검사합니다.


In [ ]:
from pathlib import Path
from collections import Counter
import json
import os
import subprocess
import sys
import time

import torch

DATA_ROOT = Path('/content/drive/MyDrive/TeamProject/test_dataset/crops_bboxfixed')
RUNS_ROOT = Path('/content/drive/MyDrive/TeamProject/test_dataset/runs')
PADDINGS = ('pad000', 'pad005', 'pad010')
CLASS_NAMES = ('clean', 'outer', 'inner')

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', torch.cuda.get_device_properties(0).total_memory / 1024**3)

data_counts = {}
all_paths_ok = True

for padding in PADDINGS:
    data_counts[padding] = {}
    print(f'\n[{padding}]')
    for split in ('train', 'val'):
        split_counts = {}
        for class_name in CLASS_NAMES:
            class_dir = DATA_ROOT / padding / split / class_name
            count = len(list(class_dir.glob('*.jpg'))) if class_dir.is_dir() else 0
            split_counts[class_name] = count
            all_paths_ok &= class_dir.is_dir()
        data_counts[padding][split] = split_counts
        print(split, split_counts, 'total=', sum(split_counts.values()))

expected_train = {'clean': 7499, 'outer': 4497, 'inner': 1738}
expected_val = {'clean': 1699, 'outer': 900, 'inner': 559}

counts_ok = all(
    data_counts[padding]['train'] == expected_train
    and data_counts[padding]['val'] == expected_val
    for padding in PADDINGS
)

print('\n경로 존재:', all_paths_ok)
print('파일 수 정상:', counts_ok)

if not all_paths_ok or not counts_ok:
    raise RuntimeError('crop 데이터 경로나 파일 수가 예상과 다릅니다.')


## 2. 학습 설정

비교 실험이므로 세 padding에 같은 설정을 적용합니다.

- `batch_size=64`: RTX 3090에서 안정적인 기준값
- `learning_rate=1e-4`: 사전학습 모델 전체 fine-tuning에 보수적인 값
- `macro F1`: 표본이 적은 inner 클래스까지 같은 비중으로 반영
- `patience=8`: macro F1이 8 epoch 연속 개선되지 않으면 조기 종료


In [ ]:
EXPERIMENT_CONFIG = {
    'paddings': list(PADDINGS),
    'class_names': list(CLASS_NAMES),
    'input_size': 224,
    'batch_size': 64,
    'num_workers': 4,
    'max_epochs': 50,
    'early_stopping_patience': 8,
    'scheduler_patience': 3,
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'seed': 42,
}

config_path = RUNS_ROOT / 'resnet18_padding_experiment_config.json'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
config_path.write_text(
    json.dumps(EXPERIMENT_CONFIG, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print(json.dumps(EXPERIMENT_CONFIG, ensure_ascii=False, indent=2))
print('저장:', config_path)


## 3. 학습 스크립트 생성

노트북 출력이 너무 길어 멈추는 것을 방지하기 위해 실제 학습은 별도 Python 프로세스에서 진행합니다.
로그는 파일로 저장하고 아래 모니터링 셀에서 최근 부분만 확인합니다.


In [ ]:
TRAINING_SCRIPT = r'''from pathlib import Path
from collections import Counter
import copy
import csv
import json
import os
import random
import time

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import ImageFile
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

ImageFile.LOAD_TRUNCATED_IMAGES = True

DATA_ROOT = Path('/content/drive/MyDrive/TeamProject/test_dataset/crops_bboxfixed')
RUNS_ROOT = Path('/content/drive/MyDrive/TeamProject/test_dataset/runs')
PADDINGS = ('pad000', 'pad005', 'pad010')
CLASS_NAMES = ('clean', 'outer', 'inner')

INPUT_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 4
MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 8
SCHEDULER_PATIENCE = 3
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
SEED = 42
MIN_DELTA = 1e-5


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def remap_imagefolder(dataset):
    desired = {'clean': 0, 'outer': 1, 'inner': 2}
    old_index_to_name = {index: name for name, index in dataset.class_to_idx.items()}

    remapped_samples = [
        (path, desired[old_index_to_name[target]])
        for path, target in dataset.samples
    ]
    dataset.samples = remapped_samples
    dataset.imgs = remapped_samples
    dataset.targets = [target for _, target in remapped_samples]
    dataset.classes = list(CLASS_NAMES)
    dataset.class_to_idx = desired
    return dataset


def make_datasets(data_dir):
    train_transform = transforms.Compose([
        transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    val_transform = transforms.Compose([
        transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    train_dataset = remap_imagefolder(
        datasets.ImageFolder(data_dir / 'train', transform=train_transform)
    )
    val_dataset = remap_imagefolder(
        datasets.ImageFolder(data_dir / 'val', transform=val_transform)
    )
    return train_dataset, val_dataset


def make_loaders(train_dataset, val_dataset):
    generator = torch.Generator()
    generator.manual_seed(SEED)

    common = {
        'batch_size': BATCH_SIZE,
        'num_workers': NUM_WORKERS,
        'pin_memory': True,
        'persistent_workers': NUM_WORKERS > 0,
        'worker_init_fn': seed_worker,
    }
    train_loader = DataLoader(
        train_dataset,
        shuffle=True,
        generator=generator,
        **common,
    )
    val_loader = DataLoader(
        val_dataset,
        shuffle=False,
        **common,
    )
    return train_loader, val_loader


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0
    labels_all = []
    predictions_all = []

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            predictions = outputs.argmax(dim=1)

            total_loss += loss.item() * images.size(0)
            total += labels.size(0)
            correct += (predictions == labels).sum().item()
            labels_all.extend(labels.cpu().numpy().tolist())
            predictions_all.extend(predictions.cpu().numpy().tolist())

    report = classification_report(
        labels_all,
        predictions_all,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    return {
        'loss': total_loss / total,
        'accuracy': correct / total,
        'macro_f1': report['macro avg']['f1-score'],
        'report': report,
        'labels': labels_all,
        'predictions': predictions_all,
    }


def save_confusion_matrices(labels, predictions, run_dir):
    matrix = confusion_matrix(labels, predictions, labels=[0, 1, 2])
    matrix_df = pd.DataFrame(matrix, index=CLASS_NAMES, columns=CLASS_NAMES)
    matrix_df.to_csv(run_dir / 'best_confusion_matrix.csv', encoding='utf-8')

    row_sums = matrix.sum(axis=1, keepdims=True)
    normalized = np.divide(
        matrix,
        row_sums,
        out=np.zeros_like(matrix, dtype=float),
        where=row_sums != 0,
    )
    normalized_df = pd.DataFrame(normalized, index=CLASS_NAMES, columns=CLASS_NAMES)
    normalized_df.to_csv(
        run_dir / 'best_confusion_matrix_normalized.csv',
        encoding='utf-8',
    )

    for values, title, filename, fmt in (
        (matrix, 'Confusion Matrix', 'best_confusion_matrix.png', 'd'),
        (
            normalized,
            'Normalized Confusion Matrix',
            'best_confusion_matrix_normalized.png',
            '.3f',
        ),
    ):
        figure, axis = plt.subplots(figsize=(7, 6))
        image = axis.imshow(values, cmap='Blues')
        figure.colorbar(image, ax=axis)
        axis.set_xticks(range(3), labels=CLASS_NAMES)
        axis.set_yticks(range(3), labels=CLASS_NAMES)
        axis.set_xlabel('Predicted')
        axis.set_ylabel('True')
        axis.set_title(title)

        threshold = float(values.max()) / 2 if values.size else 0
        for row in range(3):
            for column in range(3):
                value = values[row, column]
                text = format(value, fmt)
                axis.text(
                    column,
                    row,
                    text,
                    ha='center',
                    va='center',
                    color='white' if value > threshold else 'black',
                )
        figure.tight_layout()
        figure.savefig(run_dir / filename, dpi=160, bbox_inches='tight')
        plt.close(figure)


def train_one_padding(padding, device):
    set_seed(SEED)
    data_dir = DATA_ROOT / padding
    run_dir = RUNS_ROOT / f'resnet18_dirty3_bboxfixed_{padding}'
    run_dir.mkdir(parents=True, exist_ok=True)

    completed_path = run_dir / 'completed.json'
    if completed_path.is_file():
        print(f'[{padding}] completed.json 존재 — 완료된 실험을 건너뜁니다.', flush=True)
        return

    train_dataset, val_dataset = make_datasets(data_dir)
    train_loader, val_loader = make_loaders(train_dataset, val_dataset)

    train_counts = np.bincount(train_dataset.targets, minlength=3)
    class_weights = len(train_dataset) / (3.0 * train_counts)
    class_weights_tensor = torch.tensor(
        class_weights,
        dtype=torch.float32,
        device=device,
    )

    print(f'\\n===== {padding} 학습 시작 =====', flush=True)
    print('class_to_idx:', train_dataset.class_to_idx, flush=True)
    print('train counts:', train_counts.tolist(), flush=True)
    print('class weights:', class_weights.tolist(), flush=True)

    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 3)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        patience=SCHEDULER_PATIENCE,
        factor=0.5,
    )

    history = []
    best_macro_f1 = -1.0
    patience_count = 0
    start_epoch = 1
    last_checkpoint_path = run_dir / 'last_checkpoint.pt'

    if last_checkpoint_path.is_file():
        checkpoint = torch.load(
            last_checkpoint_path,
            map_location=device,
            weights_only=False,
        )
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        history = checkpoint['history']
        best_macro_f1 = checkpoint['best_macro_f1']
        patience_count = checkpoint['patience_count']
        start_epoch = checkpoint['epoch'] + 1
        print(f'[{padding}] epoch {start_epoch}부터 resume', flush=True)

    experiment_start = time.time()

    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        total_loss = 0.0
        total = 0
        correct = 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            predictions = outputs.argmax(dim=1)
            total_loss += loss.item() * images.size(0)
            total += labels.size(0)
            correct += (predictions == labels).sum().item()

        train_loss = total_loss / total
        train_accuracy = correct / total
        validation = evaluate(model, val_loader, criterion, device)
        scheduler.step(validation['macro_f1'])
        current_lr = optimizer.param_groups[0]['lr']

        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'train_accuracy': train_accuracy,
            'val_loss': validation['loss'],
            'val_accuracy': validation['accuracy'],
            'macro_f1': validation['macro_f1'],
            'clean_precision': validation['report']['clean']['precision'],
            'clean_recall': validation['report']['clean']['recall'],
            'clean_f1': validation['report']['clean']['f1-score'],
            'outer_precision': validation['report']['outer']['precision'],
            'outer_recall': validation['report']['outer']['recall'],
            'outer_f1': validation['report']['outer']['f1-score'],
            'inner_precision': validation['report']['inner']['precision'],
            'inner_recall': validation['report']['inner']['recall'],
            'inner_f1': validation['report']['inner']['f1-score'],
            'learning_rate': current_lr,
            'epoch_seconds': time.time() - epoch_start,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)

        improved = validation['macro_f1'] > best_macro_f1 + MIN_DELTA
        if improved:
            best_macro_f1 = validation['macro_f1']
            patience_count = 0

            # API와 단순 로딩에 사용할 raw state_dict
            torch.save(
                model.state_dict(),
                run_dir / 'best_resnet18_dirty3_state_dict.pt',
            )

            # 재현과 메타데이터 보존용 checkpoint
            best_checkpoint = {
                'architecture': 'resnet18',
                'padding': padding,
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'class_names': list(CLASS_NAMES),
                'class_to_idx': {name: index for index, name in enumerate(CLASS_NAMES)},
                'input_size': INPUT_SIZE,
                'normalization_mean': [0.485, 0.456, 0.406],
                'normalization_std': [0.229, 0.224, 0.225],
                'best_macro_f1': best_macro_f1,
                'val_accuracy': validation['accuracy'],
                'class_weights': class_weights.tolist(),
            }
            torch.save(
                best_checkpoint,
                run_dir / 'best_resnet18_dirty3_checkpoint.pt',
            )

            best_metrics = {
                'padding': padding,
                'epoch': epoch,
                'val_loss': validation['loss'],
                'val_accuracy': validation['accuracy'],
                'macro_f1': validation['macro_f1'],
                'classification_report': validation['report'],
                'class_names': list(CLASS_NAMES),
                'class_to_idx': {name: index for index, name in enumerate(CLASS_NAMES)},
                'train_counts': train_counts.tolist(),
                'class_weights': class_weights.tolist(),
            }
            (run_dir / 'best_metrics.json').write_text(
                json.dumps(best_metrics, ensure_ascii=False, indent=2),
                encoding='utf-8',
            )
            save_confusion_matrices(
                validation['labels'],
                validation['predictions'],
                run_dir,
            )
        else:
            patience_count += 1

        torch.save(
            {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'history': history,
                'best_macro_f1': best_macro_f1,
                'patience_count': patience_count,
                'padding': padding,
                'class_names': list(CLASS_NAMES),
            },
            last_checkpoint_path,
        )

        print(
            f'[{padding}] epoch {epoch:02d}/{MAX_EPOCHS} | '
            f'train loss {train_loss:.4f} acc {train_accuracy:.4f} | '
            f'val loss {validation["loss"]:.4f} acc {validation["accuracy"]:.4f} | '
            f'macro F1 {validation["macro_f1"]:.4f} | '
            f'clean {validation["report"]["clean"]["f1-score"]:.4f} '
            f'outer {validation["report"]["outer"]["f1-score"]:.4f} '
            f'inner {validation["report"]["inner"]["f1-score"]:.4f} | '
            f'lr {current_lr:.2e} | patience {patience_count}/{EARLY_STOPPING_PATIENCE}',
            flush=True,
        )

        if patience_count >= EARLY_STOPPING_PATIENCE:
            print(f'[{padding}] Early stopping at epoch {epoch}', flush=True)
            break

    completed = {
        'padding': padding,
        'status': 'completed',
        'best_macro_f1': best_macro_f1,
        'last_epoch': history[-1]['epoch'],
        'elapsed_minutes': (time.time() - experiment_start) / 60,
        'best_state_dict': str(run_dir / 'best_resnet18_dirty3_state_dict.pt'),
        'best_checkpoint': str(run_dir / 'best_resnet18_dirty3_checkpoint.pt'),
    }
    completed_path.write_text(
        json.dumps(completed, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    print(f'[{padding}] 완료: {completed}', flush=True)

    del model, optimizer, scheduler, train_loader, val_loader
    torch.cuda.empty_cache()


def main():
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPU를 사용할 수 없습니다.')
    device = torch.device('cuda:0')
    print('device:', device, torch.cuda.get_device_name(0), flush=True)

    for padding in PADDINGS:
        train_one_padding(padding, device)

    print('\\n모든 padding 학습 완료', flush=True)


if __name__ == '__main__':
    main()
'''

SCRIPT_PATH = Path('/content/drive/MyDrive/TeamProject/test_dataset/train_resnet18_padding_experiment.py')
SCRIPT_PATH.write_text(TRAINING_SCRIPT, encoding='utf-8')
print('저장:', SCRIPT_PATH)
compile(TRAINING_SCRIPT, str(SCRIPT_PATH), 'exec')
print('학습 스크립트 문법 검사 통과')


In [ ]:
import sys
import sklearn

print("Python:", sys.executable)
print("scikit-learn:", sklearn.__version__)

In [ ]:
from pathlib import Path

RUNS_ROOT = Path("/content/drive/MyDrive/TeamProject/test_dataset/runs")
PID_PATH = (
    RUNS_ROOT
    / "resnet18_padding_experiment.pid"
)
LOG_PATH = (
    RUNS_ROOT
    / "resnet18_padding_experiment.log"
)

# 현재 노트북에 process 객체가 남았다면 종료 상태 회수
if "process" in globals():
    return_code = process.poll()
    print("이전 프로세스 종료 코드:", return_code)

# 종료된 프로세스의 PID 기록 제거
if PID_PATH.is_file():
    old_pid = PID_PATH.read_text().strip()
    PID_PATH.unlink()
    print("종료된 PID 파일 제거:", old_pid)
else:
    print("PID 파일 없음")

In [ ]:
FAILED_LOG_PATH = (
    RUNS_ROOT
    / "resnet18_padding_experiment_failed_sklearn.log"
)

if LOG_PATH.is_file():
    if FAILED_LOG_PATH.is_file():
        FAILED_LOG_PATH.unlink()

    LOG_PATH.rename(FAILED_LOG_PATH)
    print("기존 오류 로그 보존:", FAILED_LOG_PATH)
else:
    print("기존 로그 없음")

## 4. 백그라운드 학습 시작

한 번만 실행하세요. `pad000 → pad005 → pad010` 순서로 학습합니다.
완료된 padding에 `completed.json`이 있으면 재실행 시 자동으로 건너뜁니다.


In [ ]:
LOG_PATH = RUNS_ROOT / 'resnet18_padding_experiment.log'
PID_PATH = RUNS_ROOT / 'resnet18_padding_experiment.pid'

existing_pid = None
if PID_PATH.is_file():
    try:
        existing_pid = int(PID_PATH.read_text().strip())
        os.kill(existing_pid, 0)
        print('이미 실행 중인 PID:', existing_pid)
    except (ValueError, ProcessLookupError, PermissionError):
        existing_pid = None

if existing_pid is None:
    log_file = LOG_PATH.open('a', encoding='utf-8')
    process = subprocess.Popen(
        [sys.executable, '-u', '/content/drive/MyDrive/TeamProject/test_dataset/train_resnet18_padding_experiment.py'],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        cwd='/content/drive/MyDrive/TeamProject/test_dataset',
        start_new_session=True,
    )
    PID_PATH.write_text(str(process.pid), encoding='utf-8')
    print('학습 시작 PID:', process.pid)
    print('로그:', LOG_PATH)


## 5. 진행 상황 확인

이 셀은 필요할 때 반복 실행합니다. 최근 로그 40줄만 출력합니다.


In [ ]:
pid = int(PID_PATH.read_text().strip()) if PID_PATH.is_file() else None

running = False
state = None
if pid is not None:
    status = subprocess.run(
        ['ps', '-o', 'stat=', '-p', str(pid)],
        capture_output=True,
        text=True,
    )
    state = status.stdout.strip()
    running = bool(state) and not state.startswith('Z')

print('PID:', pid)
print('process state:', state)
print('실행 중:', running)

if LOG_PATH.is_file():
    lines = LOG_PATH.read_text(encoding='utf-8', errors='replace').splitlines()
    print('\n===== 최근 로그 40줄 =====')
    print('\n'.join(lines[-40:]))
else:
    print('로그 파일이 아직 없습니다.')


## 6. GPU 사용 상태 확인

학습 프로세스가 실제로 GPU를 사용하는지 확인합니다.


In [ ]:
subprocess.run(['nvidia-smi'])


## 7. 학습 완료 후 세 padding 결과 비교

모든 학습이 끝난 뒤 실행합니다. validation macro F1을 1차 선정 기준으로 사용합니다.


In [ ]:
import pandas as pd

comparison_rows = []

for padding in PADDINGS:
    run_dir = RUNS_ROOT / f'resnet18_dirty3_bboxfixed_{padding}'
    metrics_path = run_dir / 'best_metrics.json'

    if not metrics_path.is_file():
        print(f'{padding}: 아직 best_metrics.json 없음')
        continue

    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    report = metrics['classification_report']
    comparison_rows.append({
        'padding': padding,
        'best_epoch': metrics['epoch'],
        'val_accuracy': metrics['val_accuracy'],
        'macro_f1': metrics['macro_f1'],
        'clean_precision': report['clean']['precision'],
        'clean_recall': report['clean']['recall'],
        'clean_f1': report['clean']['f1-score'],
        'outer_precision': report['outer']['precision'],
        'outer_recall': report['outer']['recall'],
        'outer_f1': report['outer']['f1-score'],
        'inner_precision': report['inner']['precision'],
        'inner_recall': report['inner']['recall'],
        'inner_f1': report['inner']['f1-score'],
    })

comparison = pd.DataFrame(comparison_rows)
if not comparison.empty:
    comparison = comparison.sort_values('macro_f1', ascending=False).reset_index(drop=True)
    display(comparison)
    comparison_path = RUNS_ROOT / 'resnet18_padding_comparison.csv'
    comparison.to_csv(comparison_path, index=False)
    print('\n비교표 저장:', comparison_path)
    print('선정 후보:', comparison.iloc[0]['padding'])
else:
    print('완료된 결과가 없습니다.')


## 8. 해석 시 주의점

- accuracy만 보고 padding을 선택하지 않습니다. 데이터가 불균형하므로 macro F1과 클래스별 F1을 함께 봅니다.
- validation에서 근소한 차이만 있다면 더 큰 padding을 무조건 선택하지 않습니다.
- 최종 2-stage 성능은 GT crop 분류 성능이 아니라 YOLO 예측 bbox → crop → ResNet18 전체 파이프라인으로 다시 평가해야 합니다.
- `best_resnet18_dirty3_state_dict.pt`는 API 로딩용이며 클래스 순서는 반드시 `clean, outer, inner`입니다.
